<a href="https://colab.research.google.com/github/asta3002/FISH/blob/main/Custom_TensorNEAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installation

In [ ]:
!pip install git+https://github.com/EMI-Group/tensorneat.git

  Cloning https://github.com/EMI-Group/tensorneat.git to /tmp/pip-req-build-qf_yi0sv
  Running command git clone --filter=blob:none --quiet https://github.com/EMI-Group/tensorneat.git /tmp/pip-req-build-qf_yi0sv
  Resolved https://github.com/EMI-Group/tensorneat.git to commit b9778dcf8b2db5f3698b3965073c2181608c467c


In [ ]:
!pip install -U "jax[tpu]==0.4.28"

In [ ]:
!pip install evojax

#Libraries

In [4]:
# Import necessary modules
import jax
import jax.numpy as jnp
import argparse
import os
from jax import random
from evojax.task.slimevolley import SlimeVolley
from tensorneat.problem import BaseProblem
from tensorneat.pipeline import Pipeline
from tensorneat.algorithm.neat import NEAT
from tensorneat.genome import DefaultGenome, BiasNode, DefaultConn,DefaultMutation
from tensorneat.genome.operations import DefaultCrossover, DefaultDistance, DefaultMutation
from tensorneat.common import ACT, AGG
from typing import Tuple
from functools import partial

#DEMO

In [ ]:
max_steps = 3000
task = SlimeVolley(test=True, max_steps=max_steps)
task_reset_fn = jax.jit(task.reset)
task_step_fn = jax.jit(task.step)
og_key = jax.random.PRNGKey(12346)[None,:]
task_state = task_reset_fn(og_key)
test_state = task.reset(og_key)
# dummy_action = jnp.array([[0.0, 1.0, -1.0]], dtype=jnp.float32)  # shape [1, 3]
# # next_state,reward,done = task_step_fn(task_state,dummy_action)
# # action = (network_output > 0.0).astype(jnp.int32)


In [ ]:
print(test_state.game_state.agent_right)

AgentState(direction=Array([1], dtype=int32, weak_type=True), x=Array([12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(test_state.game_state.agent_left)

AgentState(direction=Array([-1], dtype=int32, weak_type=True), x=Array([-12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(test_state.game_state.ball)

ParticleState(x=Array([0.], dtype=float32), y=Array([12.], dtype=float32), prev_x=Array([0.], dtype=float32), prev_y=Array([12.], dtype=float32), vx=Array([6.860523], dtype=float32), vy=Array([20.072697], dtype=float32), r=Array([0.5], dtype=float32))


In [ ]:
def squeeze_state(state):
        return jax.tree_util.tree_map(lambda x: jnp.squeeze(x, axis=0) if hasattr(x, "ndim") and x.ndim > 0 else x, state)

In [ ]:
# task.render(squeeze_state(test_state))

In [ ]:
print(task_state.obs)

[[1.2       0.15      0.        0.        0.        1.2       0.6860523
  2.0072696 1.2       0.15      0.        0.       ]]


In [ ]:
print(task_state.game_state.agent_left)

AgentState(direction=Array([-1], dtype=int32, weak_type=True), x=Array([-12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(task_state.game_state.agent_right)

AgentState(direction=Array([1], dtype=int32, weak_type=True), x=Array([12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(task_state.game_state.ball)

ParticleState(x=Array([0.], dtype=float32), y=Array([12.], dtype=float32), prev_x=Array([0.], dtype=float32), prev_y=Array([12.], dtype=float32), vx=Array([6.860523], dtype=float32), vy=Array([20.072697], dtype=float32), r=Array([0.5], dtype=float32))


In [ ]:
# print(task_state.game_state.agent_left.life,task_state.game_state.agent_right.life)

In [ ]:
# next_state = task_state

In [ ]:
# total_reward = jnp.array(0.0, dtype=jnp.float32)

In [ ]:
# for i in range(3000):
#   next_state,reward,done = task_step_fn(next_state,dummy_action)
#   reward = jnp.squeeze(reward).astype(jnp.float32)
#   if(reward):
#     print(reward,done,next_state.game_state.agent_left.life,next_state.game_state.agent_right.life)
#   total_reward = total_reward + reward
#   if done:
#     break




In [ ]:
# print(total_reward)

In [ ]:
# print(task_state.game_state)
# print(task_state.obs)

#SlimeVolleyBall Environment

In [5]:
import jax.numpy as jnp

def adjusted_fitness(s_fitness):
    """
    Calculates the total adjusted fitness for a species, as described in the original NEAT paper.

    Args:
        s_fitness (jnp.ndarray): A JAX array containing the fitness values of individuals
                                 in a single species. Non-members are represented by -jnp.inf.

    Returns:
        jnp.ndarray: The total adjusted fitness of the species.
    """
    # 1. Create a boolean mask to identify valid fitness values (i.e., not -jnp.inf)
    is_member_mask = jnp.isfinite(s_fitness)

    # 2. Count the number of members in the species
    species_size = jnp.sum(is_member_mask)

    # Use a conditional to handle the case of an empty species (size=0) to avoid division by zero.
    species_size = jnp.where(species_size == 0, 1, species_size)

    # 3. Adjust the fitness of each member by dividing by the species size.
    # The where clause ensures only valid fitnesses are adjusted.
    adjusted_s_fitness = jnp.where(is_member_mask, s_fitness / species_size, 0.0)

    # 4. Sum the adjusted fitness values to get the species' total adjusted fitness.
    total_adjusted_fitness = jnp.sum(adjusted_s_fitness)

    return total_adjusted_fitness

In [16]:
VELOCITY_THRESHOLD = 18.0

In [27]:
class SlimeVolleyEnv(BaseProblem):
    jitable = True  # Needed for TensorNEAT JIT support

    def __init__(self, max_episode_steps=3000, num_episodes=3):
        super().__init__()
        self.max_steps = max_episode_steps
        self.episodes = num_episodes

        # Create task instance
        self.task = SlimeVolley(test=False, max_steps=self.max_steps)
        self.test_task = SlimeVolley(test=True, max_steps=3000)

        # JIT compile reset and step functions
        self.task_reset = jax.jit(self.task.reset)
        self.task_step = jax.jit(self.task.step)
        self.test_task_reset = jax.jit(self.test_task.reset)
        self.test_task_step = jax.jit(self.test_task.step)
    def reward_wrapper(self, current_state, previous_state):
        """
        JAX-native version of collision detection, optimal angle detection and velocity check.

        Args:
            current_state: Current Environment State of ball, agent, and opposition agent
            previous_state: Previous Environment State of ball, agent, and opposition agent

        Returns:
            JAX Array : Reward for the current state, based on if the ball is in a optimal angle and  above velocity threshold, after contact with agent, for increased chances of win
        """
        # Current state variables
        ball_x = current_state.game_state.ball.x
        ball_y = current_state.game_state.ball.y
        agent_x = current_state.game_state.agent_right.x
        agent_y = current_state.game_state.agent_right.y
        opp_x = current_state.game_state.agent_left.x
        opp_y = current_state.game_state.agent_left.y
        ball_vel_x = current_state.game_state.ball.vx
        ball_vel_y = current_state.game_state.ball.vy

        # Previous state variables
        ball_vel_px = previous_state.game_state.ball.vx
        ball_vel_py = previous_state.game_state.ball.vy
        ball_px = previous_state.game_state.ball.x
        ball_py = previous_state.game_state.ball.y
        agent_px = previous_state.game_state.agent_right.x
        agent_py = previous_state.game_state.agent_right.y

        # Define parameters

        ball_radius = current_state.game_state.ball.r[0]
        agent_radius = current_state.game_state.agent_right.r[0]

        # Target point coordinates
        target_x = 0.0
        target_y = 5.0

        # Check if ball made contact with agent in previous state
        distance_prev = (ball_px - agent_px)**2 + (ball_py - agent_py)**2
        distance_threshold = (ball_radius + agent_radius)*(ball_radius + agent_radius)
        contact_made = jnp.abs(distance_prev - (ball_radius + agent_radius)) < 1e-6  # Small tolerance for floating point
        contact_made_with_jump = contact_made & (ball_py>1.5)
        # Calculate current ball velocity magnitude
        ball_speed = ball_vel_x**2 + ball_vel_y**2

        # Calculate current angle of motion of the ball (in radians)
        current_angle = jnp.arctan2(ball_vel_y, ball_vel_x)

        # Calculate optimal angle needed to reach target point (0, 3)
        dx_to_target = target_x - ball_x
        dy_to_target = target_y - ball_y
        optimal_angle = jnp.arctan2(dy_to_target, dx_to_target)

        # Horizontal angle pointing left (-180 degrees = -π radians)
        horizontal_angle = -jnp.pi

        # Check velocity condition
        velocity_above_threshold = ball_speed > VELOCITY_THRESHOLD



        # For third quadrant movement (left and potentially up/down):
        # Current angle should be between optimal angle and horizontal (-π)
        # Since we're in third quadrant, angles are between -π and -π/2
        angle_condition_1 = (current_angle < optimal_angle) & (current_angle > horizontal_angle)

        # Check if current angle is above horizontal (moving leftward and upward toward opponent)
        angle_condition_2 = (current_angle > jnp.pi/2) & (current_angle < jnp.pi)

        # Calculate reward only if contact was made in previous state
        # conditional_reward = jnp.where(current_state.steps<200,10,jnp.where(current_state.steps<400,5,2))
        conditional_reward =0.1
        reward_value = jnp.where(
            angle_condition_1 & velocity_above_threshold,
            conditional_reward,  # Optimal angle range with high velocity
            jnp.where(
                angle_condition_1 & ~velocity_above_threshold,
                conditional_reward/2,  # Optimal angle range with low velocity
                jnp.where(
                    angle_condition_2 & velocity_above_threshold,
                    0.05,  # Above horizontal with high velocity
                    0.0   # All other cases
                )
            )
        )

        # Return reward only if contact was made, otherwise return 0.0
        reward = jnp.where(contact_made_with_jump, reward_value, 0.0)

        return reward
    def evaluate(self, state, randkey, act_func, params):
        """Evaluate an individual on SlimeVolley."""

        def run_episode(ep_key):
            # Reset environment
            reset_key = ep_key[None, :]
            task_state = self.task_reset(reset_key)

            # Carry: (task_state, total_reward, randkey)
            sum_reward = jnp.array(0.0, dtype=jnp.float32)
            init_carry = (task_state, sum_reward)
            # @jax.jit
            def step_fn(carry, _):
                task_state, total_reward = carry
                # key, step_key = jax.random.split(key)
                # print(task_state)

                # obs = self.task.get_obs(task_state, self.task.params)
                obs  = task_state.obs
                # obs = obs * obs.dtype.type(10)
                obs = obs.squeeze()
                # print(obs.ndim)

                action_out = act_func(task_state, params,obs)


                # Convert output to discrete action (argmax over action space)
                # jax.debug.print("action_out: {}", action_out)
                action = (action_out > 0.0).astype(jnp.float32)
                # jax.debug.print("action_out: {}", action)
                jax_action = action[None,:]
                # jax.debug.print("action_out: {}", jax_action)
                # jax.debug.print("task_state : {}",task_state)
                # jax.debug.print("action_out: {}", jax_action)
                new_state, reward, done = self.task_step(task_state, jax_action)
                reward = jnp.where(reward>0,15,jnp.where(reward<0,-0.05,0))
                reward = jnp.squeeze(reward).astype(jnp.float32)
                # reward = jnp.where(reward > 0, reward.dtype.type(15), reward)
                bonus_reward= self.reward_wrapper(new_state,task_state)
                bonus_reward = jnp.squeeze(bonus_reward).astype(jnp.float32)
                # After uncommenting add bonus_reward to total_reward



                # reward = jnp.where(reward == 0,reward.dtype.type(-0.5),reward)
                # jax.debug.print(reward)
                total_reward = total_reward + reward+bonus_reward

                return (new_state, total_reward), None

            final_carry, _ = jax.lax.scan(step_fn, init_carry, None, length=self.max_steps)
            return final_carry[1]  # total_reward

        # Run multiple episodes and average reward
        ep_keys = jax.random.split(randkey, self.episodes)
        rewards = jax.vmap(run_episode)(ep_keys)
        return jnp.mean(rewards)

    @property
    def input_shape(self):
        return self.task.obs_shape  # Provided by SlimeVolley

    @property
    def output_shape(self):
        return self.task.act_shape  # Provided by SlimeVolley

    def squeeze_state(self,state):
        return jax.tree_util.tree_map(lambda x: jnp.squeeze(x, axis=0) if hasattr(x, "ndim") and x.ndim > 0 else x, state)

    def show(self, state, randkey, act_func, params, *args, **kwargs):

        """Render a few episodes to visually inspect performance."""
        import time
        import numpy as np  # Add numpy import for type conversion
        print("\n" + "="*50)
        print("SLIMEVOLLEY AGENT PERFORMANCE")
        print("="*50)

        best_reward = float('-inf')
        best_screens = []
        best_steps = 0

        # JAX-compiled function for running a single episode
        @jax.jit
        def run_single_episode(episode_key):
            """Run a single episode and return states, rewards, and done flags."""
            test_task_state = self.test_task_reset(episode_key[None, :])

            def step_fn(carry, _):
                task_state, total_reward, step_count, terminated = carry

                # Only process if not terminated
                obs = task_state.obs.squeeze()
                action_out = act_func(task_state, params, obs)
                action = (action_out > 0.0).astype(jnp.float32)
                jax_action = action[None, :]

                new_state, reward, done = self.test_task_step(task_state, jax_action)
                reward = jnp.squeeze(reward).astype(jnp.float32)
                done = jnp.squeeze(done).astype(jnp.bool_)
                # Update only if not already terminated
                new_total_reward = jnp.where(terminated, total_reward, total_reward + reward)
                new_step_count = jnp.where(terminated, step_count, step_count + 1)
                new_terminated = terminated | done

                # Use previous state if already terminated
                final_state = jax.tree_util.tree_map(
                    lambda old, new: jnp.where(terminated, old, new),
                    task_state, new_state
                )

                return (final_state, new_total_reward, new_step_count, new_terminated), {
                    'state': final_state,
                    'reward': reward,
                    'done': done,
                    'terminated': new_terminated
                }

            init_carry = (test_task_state, jnp.array(0.0, dtype=jnp.float32),
                        jnp.array(0, dtype=jnp.int32), jnp.array(False))

            final_carry, history = jax.lax.scan(step_fn, init_carry, None, length=3000)

            return {
                'final_reward': final_carry[1],
                'final_steps': final_carry[2],
                'states': history['state'],
                'rewards': history['reward'],
                'dones': history['done'],
                'terminated_flags': history['terminated']
            }

        # Run episodes using JAX
        episode_keys = jax.random.split(randkey, 3)

        for i in range(3):
            # Run episode with JAX
            episode_result = run_single_episode(episode_keys[i])

            total_reward = float(episode_result['final_reward'])
            final_steps = int(episode_result['final_steps'])
            states = episode_result['states']
            rewards = episode_result['rewards']
            dones = episode_result['dones']
            terminated_flags = episode_result['terminated_flags']

            # Find the actual end of the episode
            done_indices = jnp.where(dones)[0]
            if len(done_indices) > 0:
                actual_end_step = int(done_indices[0]) + 1
                print(f"{done_indices[0]}, {total_reward}, {float(rewards[done_indices[0]])}, "
                      f"{states.game_state.agent_left.life[done_indices[0]]}, "
                      f"{states.game_state.agent_right.life[done_indices[0]]}, {True}")
            else:
                actual_end_step = 3000

            print(f"Episode {i+1} Reward: {total_reward}")

            # Generate screens for this episode (only up to actual end)
            current_screens = []
            for step in range(actual_end_step):
                # Extract state at this step and squeeze it
                step_state = jax.tree_util.tree_map(lambda x: x[step], states)
                render_state = self.squeeze_state(step_state)
                current_screens.append(self.test_task.render(render_state))

            # Check if this is the best episode so far
            if total_reward > best_reward:
                best_reward = total_reward
                best_screens = current_screens.copy()
                best_steps = actual_end_step
            elif total_reward == best_reward:
                if actual_end_step > best_steps:
                    best_reward = total_reward
                    best_screens = current_screens.copy()
                    best_steps = actual_end_step

        print(f"\nBest episode reward: {best_reward}")

        # Create GIF only from the best episode
        log_dir = '/content'
        gif_file = os.path.join(log_dir, 'slimevolley_best.gif')
        if best_screens:
            best_screens[0].save(gif_file, save_all=True, append_images=best_screens[1:],
                                duration=40, loop=0)
            print(f"GIF created from best episode and saved to: {gif_file}")
        else:
            print("No screens to save for GIF")
    # def show(self, state, randkey, act_func, params, *args, **kwargs):

    #     """Render a few episodes to visually inspect performance."""
    #     import time
    #     print("\n" + "="*50)
    #     print("SLIMEVOLLEY AGENT PERFORMANCE")
    #     print("="*50)

    #     best_reward = float('-inf')
    #     best_screens = []
    #     # episode_rewards = []
    #     best_steps = 0

    #     for i in range(3):
    #         key, randkey = jax.random.split(randkey)
    #         test_task_state = self.test_task.reset(key[None, :])
    #         total_reward = jnp.array(0.0, dtype=jnp.float32)
    #         current_screens = []
    #         current_step=0
    #         for step in range(3000):
    #             obs = test_task_state.obs
    #             obs = obs.squeeze()
    #             action_out = act_func(test_task_state, params, obs)

    #             # Convert output to discrete action (argmax over action space)
    #             action = (action_out > 0.0).astype(jnp.float32)
    #             jax_action = action[None, :]

    #             new_state, reward, done = self.test_task.step(test_task_state, jax_action)
    #             render_state = self.squeeze_state(new_state)

    #             # Store screen for current episode
    #             current_screens.append(self.test_task.render(render_state))

    #             test_task_state = new_state
    #             # if reward:
    #             #     print(total_reward, reward, test_task_state.game_state.agent_left.life,test_task_state.game_state.agent_right.life, done)
    #             total_reward = total_reward + reward
    #             current_step+=1
    #             if done:
    #                 print(step,total_reward, reward, test_task_state.game_state.agent_left.life,test_task_state.game_state.agent_right.life, done)
    #                 break

    #         # episode_rewards.append(float(total_reward))
    #         print(f"Episode {i+1} Reward: {total_reward}")

    #         # Check if this is the best episode so far
    #         if total_reward > best_reward:
    #             best_reward = total_reward
    #             best_screens = current_screens.copy()  # Save screens from best episode
    #             best_steps = current_step
    #         elif total_reward == best_reward:
    #               if(current_step> best_steps):
    #                 best_reward = total_reward
    #                 best_screens = current_screens.copy()



    #     print(f"\nBest episode reward: {best_reward}")

    #     # Create GIF only from the best episode
    #     log_dir = '/content'
    #     gif_file = os.path.join(log_dir, 'slimevolley_best.gif')
    #     best_screens[0].save(gif_file, save_all=True, append_images=best_screens[1:],
    #                         duration=40, loop=0)

    #     print(f"GIF created from best episode and saved to: {gif_file}")


    # def show(self, state, randkey, act_func, params, *args, **kwargs):

    #     """Render a few episodes to visually inspect performance."""
    #     import time
    #     print("\n" + "="*50)
    #     print("SLIMEVOLLEY AGENT PERFORMANCE")
    #     print("="*50)

    #     best_reward = float('-inf')
    #     best_screens = []
    #     # episode_rewards = []

    #     for i in range(3):
    #         key, randkey = jax.random.split(randkey)
    #         test_task_state = self.test_task.reset(key[None, :])
    #         total_reward = jnp.array(0.0, dtype=jnp.float32)
    #         current_screens = []

    #         for step in range(3000):
    #             obs = test_task_state.obs
    #             obs = obs.squeeze()
    #             action_out = act_func(test_task_state, params, obs)

    #             # Convert output to discrete action (argmax over action space)
    #             action = (action_out > 0.0).astype(jnp.float32)
    #             jax_action = action[None, :]

    #             new_state, reward, done = self.test_task.step(test_task_state, jax_action)
    #             render_state = self.squeeze_state(new_state)

    #             # Store screen for current episode
    #             current_screens.append(self.test_task.render(render_state))

    #             test_task_state = new_state
    #             # if reward:
    #             #     print(total_reward, reward, test_task_state.game_state.agent_left.life,test_task_state.game_state.agent_right.life, done)
    #             total_reward = total_reward + reward

    #             if done:
    #                 print(step, total_reward, reward, test_task_state.game_state.agent_left.life,test_task_state.game_state.agent_right.life, done)
    #                 break

    #         # episode_rewards.append(float(total_reward))
    #         print(f"Episode {i+1} Reward: {total_reward}")

    #         # Check if this is the best episode so far
    #         if total_reward > best_reward:
    #             best_reward = total_reward
    #             best_screens = current_screens.copy()  # Save screens from best episode

    #     print(f"\nBest episode reward: {best_reward}")

    #     # Create GIF only from the best episode
    #     log_dir = '/content'
    #     gif_file = os.path.join(log_dir, 'slimevolley_best.gif')
    #     best_screens[0].save(gif_file, save_all=True, append_images=best_screens[1:],
    #                         duration=40, loop=0)

    #     print(f"GIF created from best episode and saved to: {gif_file}")


#Training

In [1]:
neat_algorithm=NEAT(
        pop_size=500,  # Larger population for complex task
        species_size=500,  # Maximum number of species allowed.More species for diversity
        survival_threshold=1.0,  #  Fraction of each species (sorted by fitness) that can be selected as parents (0-1). Only the top X% survive to reproduce, creating selection pressure.
        genome=DefaultGenome(
            num_inputs=12,  # SlimeVolley observation space
            num_outputs=3,  # SlimeVolley action space (3 discrete actions)
            max_nodes=100,
            max_conns=1500,
            init_hidden_layers=(),  # Start with no hidden layers
            node_gene=OriginNode( bias_init_mean= 0.0,
                                bias_init_std = 0.1,
                                bias_mutate_power = 0.15,
                                bias_mutate_rate = 0.2,
                                bias_replace_rate = 0.015,
                                bias_lower_bound = -5,
                                bias_upper_bound = 5,
                                aggregation_default = AGG.sum,
                                aggregation_options = [AGG.sum, AGG.mean, AGG.max,AGG.product],
                                aggregation_replace_rate = 0.1,
                                activation_default = ACT.relu,
                                activation_options=[
                                                    ACT.relu,
                                                    ACT.sigmoid,
                                                    ACT.sin,
                                                    ACT.tanh,
                                                    ACT.ad
                                                  # ACT.identity,  # Can help with periodic behaviors
                                                  ],
                                activation_replace_rate= 0.1,


                                ),
            conn_gene=OriginConn(weight_init_mean  = 0.0,
                                  weight_init_std   = 1.0,
                                  weight_mutate_power = 0.2, #scale of perturbation
                                  weight_mutate_rate  = 0.2,  #probability of perturbation
                                  weight_replace_rate = 0.015,
                                  weight_lower_bound = -5,
                                  weight_upper_bound = 5,
                                  ),
            mutation=DefaultMutation(conn_add = 0.6,
                                     conn_delete  = 0.05,
                                     node_add = 0.6,
                                     node_delete = 0.05,
                                     ),
            crossover=DefaultCrossover(),
            distance=DefaultDistance(compatibility_disjoint = 1.0,
                                     compatibility_weight  = 0.4,
                                     ),
            output_transform=ACT.tanh, # Tanh output activation
            input_transform=None,


                        ),

        max_stagnation =  30, #Number of generations a species can go without improvement before being eliminated.
        species_elitism = 4, #Number of top-ranked species protected from stagnation. These species survive even if they haven't improved recently.
        spawn_number_change_rate= 0.5, # Controls how quickly species population sizes can change between generations (0-1). Smooths population changes: new_size = old_size + (target_size - old_size) * rate. Lower values = more gradual changes.
        genome_elitism  = 7, #- Number of best individuals per species that are preserved without mutation. They undergo crossover but skip mutation to preserve top performers.
        min_species_size = 1, #Minimum number of individuals a species must have. Prevents species from becoming too small and going extinct prematurely.
        compatibility_threshold = 1.0, #Maximum genetic distance for an individual to join a species. Lower values = stricter speciation (more species with similar members). Higher values = looser speciation (fewer, more diverse species
        species_fitness_func = jnp.max, #Function to compute species fitness from member fitnesses (default: jnp.max). Could also use mean, median, etc. Determines how species are ranked.
        species_number_calculate_by ="rank",#Method for allocating population across species( "rank": Linear ranking ; "fitness": Proportional to actual fitness values (higher fitness = more members) )
    )


NameError: name 'NEAT' is not defined

In [33]:
pipeline = Pipeline(
    algorithm=neat_algorithm,
    problem=SlimeVolleyEnv(
        max_episode_steps=3000,  # Shorter episodes for faster training
        num_episodes=3  # Average over 3 episodes per evaluation
    ),
    generation_limit=500,  # Run for 200 generations
    fitness_target=1000.0,  # Stop if this fitness is reached
    seed=23,
)



In [34]:
state = pipeline.setup()

state, best = pipeline.auto_run(state)

initializing
initializing finished
start compile
compile finished, cost time: 37.851104s
Generation: 1, Cost time: 4094.90ms
 	fitness: valid cnt: 500, max: 38.4500, min: -2.0167, mean: 8.8124, std: 7.4093

	node counts: max: 16, min: 15, mean: 15.47
 	conn counts: max: 38, min: 34, mean: 36.51
 	species: 151, [30, 15, 15, 18, 37, 4, 1, 3, 3, 4, 9, 5, 2, 6, 1, 2, 2, 13, 3, 1, 11, 3, 2, 1, 1, 5, 4, 4, 4, 2, 1, 3, 2, 4, 4, 14, 1, 2, 1, 1, 9, 2, 1, 1, 1, 2, 3, 15, 6, 2, 4, 2, 2, 1, 3, 1, 1, 1, 4, 3, 1, 3, 2, 1, 1, 2, 1, 1, 1, 2, 19, 3, 2, 4, 2, 2, 4, 1, 4, 1, 2, 1, 3, 1, 2, 1, 1, 9, 1, 1, 2, 1, 6, 2, 2, 2, 4, 3, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, 2, 4, 2, 2, 1, 5, 2, 1, 2, 1, 3, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 4, 1, 1, 1, 2, 1, 1, 1, 2, 1, 3, 2, 2, 2, 1, 4, 1, 2, 1, 1, 1]

Generation: 2, Cost time: 4526.63ms
 	fitness: valid cnt: 500, max: 38.5000, min: -1.9667, mean: 8.4610, std: 7.4667

	node counts: max: 17, min: 15, mean: 15.69
 	conn counts: max: 39, min: 34, mean: 36.79
 	species: 151, 

KeyboardInterrupt: 

In [ ]:
# show result
pipeline.show(state, best)

In [ ]:
network = neat_algorithm.genome.network_dict(state, *best)
neat_algorithm.genome.visualize(network, save_path="./volleyball_network.png")

In [ ]:
github_path = "First/NEAT_JAX/Custom_TensorNEAT.ipynb"